In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql. functions import col, isnull, when
from pyspark.sql. types import TimestampType


StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 5, Finished, Available, Finished, False)

In [1]:
import os

print(os.listdir("/lakehouse/default/Files"))

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 3, Finished, Available, Finished, False)

['2026-08-13_earthquake_data.json']


In [5]:
# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline","true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 7, Finished, Available, Finished, False)

In [16]:
df

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 23, Finished, Available, Finished, False)

DataFrame[longitude: double, latitude: double, elevation: double, title: string, place_description: string, sig: bigint, mag: double, magType: string, time: timestamp, updated: timestamp]

In [15]:
df.head()

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 21, Finished, Available, Finished, False)

Row(longitude=121.279, latitude=-8.3232, elevation=35.0, title='M 4.6 - 71 km NW of Ende, Indonesia', place_description='71 km NW of Ende, Indonesia', sig=326, mag=4.6, magType='mb', time=datetime.datetime(2026, 8, 18, 23, 59, 16, 815000), updated=datetime.datetime(2026, 8, 19, 4, 51, 30, 40000))

In [8]:
from pyspark.sql.functions import col

# Reshape earthquake data
df = (
    df
    .select(
        col("geometry.coordinates").getItem(0).alias("longitude"),
        col("geometry.coordinates").getItem(1).alias("latitude"),
        col("geometry.coordinates").getItem(2).alias("elevation"),
        col("properties.title").alias("title"),
        col("properties.place").alias("place_description"),
        col("properties.sig").alias("sig"),
        col("properties.mag").alias("mag"),
        col("properties.magType").alias("magType"),
        col("properties.time").alias("time"),
        col("properties.updated").alias("updated")
    )
)



StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 12, Finished, Available, Finished, False)

In [11]:
from pyspark.sql.functions import col, when, isnull

# Validate data: Check for missing or null values
df = (
    df
    .withColumn( "longitude",when(isnull(col("longitude")), 0).otherwise(col("longitude")))
    .withColumn("latitude",when(isnull(col("latitude")), 0).otherwise(col("latitude")))
    .withColumn("time",when(isnull(col("time")), 0).otherwise(col("time")))
)


StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 16, Finished, Available, Finished, False)

In [12]:
# Convert 'time' and 'updated' to timestamp
df = (
    df
    .withColumn("time", (col("time") / 1000).cast(TimestampType()))
    .withColumn("updated", (col("updated") / 1000).cast(TimestampType()))
)

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 17, Finished, Available, Finished, False)

In [14]:
# Append to the silver table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, 8c992ff6-1c7f-4101-967b-875f5ad677c1, 20, Finished, Available, Finished, False)